## Train/test + preprocessing

**Att fånga så många verkligt misstänkta fall som möjligt.**
Alltså en binär klassificering:
- 1 = farligt / misstänkt 
- 0 = inte farligt
- `is_suspicious` = 1 → farligt fall
- `is_suspicious` = 0 → normalt fall

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme()
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
X = df.drop("is_suspicious", axis=1)
y = df["is_suspicious"]

if "id" in X.columns:
    X = X.drop(columns=["id"])

print(y.value_counts(normalize=True))

In [ ]:
# Train/Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_SEED
)

print("Train proportion:\n", y_train.value_counts(normalize=True))
print("Test proportion:\n", y_test.value_counts(normalize=True))

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# Identifiera numeriska/kategoriska kolumner:

numeric_features = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("\nNumerical:", numeric_features)
print("Categorical:", categorical_features)

In [ ]:
# Preprocessing pipelines

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 4) ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ],
    remainder="drop"
)

In [ ]:
# Pipeline (no leakage) - sanity check och fit endast på train:

pipeline_sanity = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED))
])

pipeline_sanity.fit(X_train, y_train)